<a href="https://www.kaggle.com/code/taraashmittal/fine-tuning-and-testing?scriptVersionId=301426465" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Fine tuning

We fine-tune gpt-oss-120b with our custom dataset of 50 high quality solutions, hints and facts. The aim is to have the model break down the current problem into its domain and look at similar facts and solutions. Much of it copied from [Unsolth's Colab notebook on fine-tuning](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(120B)_A100-Fine-tuning.ipynb#scrollTo=QmUBVEnvCDJv).

In [1]:
import pandas as pd

data = pd.read_parquet("/kaggle/input/datasets/taraashmittal/aimo-3/dataset.parquet")
data.head()

,problem,solution,hints,facts,algebra,induction,polynomials,inequalities,geometry,trigonometry,...,abstract_algebra,group_theory,finite_fields,functional_equations,generating_functions,recurrence_relations,game_theory,optimisation,invariance,contradiction
0,A circle is divided into six consecutive secto...,"Let $a_1,\dots,a_6$ denote the numbers in the ...",1. Write down how a single move changes the si...,1. An invariant is a quantity that remains unc...,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,True
1,Let $p(x)=x^{2}-3x+2$. Prove that for every po...,First factor the divisor:\n$$p(x)=x^{2}-3x+2=(...,1. Write $p(x)$ as a product of linear factors...,1. $x^{2}-3x+2$ factors as $(x-1)(x-2)$.\n2. R...,True,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Let $n\ge 1$ be an integer and let $\omega = e...,"Denote $P(z_1,\dots ,z_n)=\det A$. By the bin...","1. Write $\det A$ as a polynomial in $z_1,\dot...",1. Binomial theorem: $(x+y)^n=\sum_{a=0}^{n}\b...,True,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,"Let $x_1,x_2,\dots ,x_k$ be real numbers such ...","Assume, for a contradiction, that at least one...",1. Replace each cosine by $(e^{i n\pi x}+e^{-i...,1. Euler’s formula: $\cos t = (e^{it}+e^{-it})...,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
4,Let $A$ and $B$ be $n\times n$ matrices over a...,Let $\lambda$ be an eigenvalue of $BA$ and let...,1. Recall that a matrix is nilpotent iff its o...,1. A matrix $M$ is nilpotent iff $M^{k}=0$ for...,True,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [4]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-120b",
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-02-25 17:36:25.247010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772040985.536495      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772040985.633033      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772040986.502719      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772040986.502755      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772040986.502757      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


Unsloth: Warning: Unknown decorator @use_kernel_forward_from_hub("MegaBlocksMoeMLP") found for GptOssMLP.
Unsloth: Warning: Unknown decorator @use_kernel_forward_from_hub("RMSNorm") found for GptOssRMSNorm.


==((====))==  Unsloth 2026.2.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    Tesla P100-PCIE-16GB. Num GPUs = 1. Max memory: 15.888 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 6.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


...

model-00013-of-00016.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00014-of-00016.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00015-of-00016.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00016-of-00016.safetensors:   0%|          | 0.00/2.14G [00:00<?, ?B/s]

NameError: name 'ParameterModule' is not defined